In [1]:
import pandas as pd

In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark_job = SparkSession.builder.appName("My Spark Regression Job").getOrCreate()

c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
df_spark = spark_job.read.option('header', 'true').csv('2015.csv', inferSchema=True)
df_spark.show(10)

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude|RateCodeID|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       2| 2015-01-15 19:05:39|  2015-01-15 19:23:42|              1|         1.59|  -73.993896484375|  40.7501106262207|         1|    

In [4]:
df_spark.write.mode("overwrite").parquet("output/taxi_parquet")

In [5]:
df_from_parquet = spark_job.read.parquet("output/taxi_parquet")

df_from_parquet.show()

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude|RateCodeID|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       1| 2015-01-13 11:46:19|  2015-01-13 12:08:42|              2|          2.6|   -74.00537109375|40.737003326416016|         1|    

In [6]:
csv_count = spark_job.read.option('header', 'true').csv('2015.csv').count()
parquet_count = spark_job.read.parquet('output/taxi_parquet').count()
print(csv_count == parquet_count)

True


In [6]:
df_spark.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RateCodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)



In [7]:
from pyspark.ml.feature import VectorAssembler

In [8]:
feature_columns = VectorAssembler(inputCols=['trip_distance', 'passenger_count'], outputCol='Independent Features')

In [9]:
output = feature_columns.transform(df_spark)
output.show(10)

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude|RateCodeID|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|Independent Features|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       2| 2015-01-15 19:05:39|  2015-01-15 19:23:42|              1|    

In [10]:
training_data = output.select('Independent Features', 'fare_amount')
training_data.show(10)

+--------------------+-----------+
|Independent Features|fare_amount|
+--------------------+-----------+
|          [1.59,1.0]|       12.0|
|           [3.3,1.0]|       14.5|
|           [1.8,1.0]|        9.5|
|           [0.5,1.0]|        3.5|
|           [3.0,1.0]|       15.0|
|           [9.0,1.0]|       27.0|
|           [2.2,1.0]|       14.0|
|           [0.8,3.0]|        7.0|
|          [18.2,3.0]|       52.0|
|           [0.9,2.0]|        6.5|
+--------------------+-----------+
only showing top 10 rows


In [12]:
from pyspark.ml.regression import LinearRegression
## Train Test Split
train_data, test_data = training_data.randomSplit([0.75, 0.25])
lr = LinearRegression(featuresCol='Independent Features', labelCol='fare_amount')
lr_model = lr.fit(train_data)

In [13]:
lr_model.coefficients

DenseVector([0.0, 0.0567])

In [14]:
lr_model.intercept

11.810353679693337

In [15]:
pred_results = lr_model.evaluate(test_data)
pred_results.predictions.show(10)

+--------------------+-----------+------------------+
|Independent Features|fare_amount|        prediction|
+--------------------+-----------+------------------+
|           [0.0,0.0]|        0.0|11.810353679693337|
|           [0.0,0.0]|        1.0|11.810353679693337|
|           [0.0,0.0]|       10.0|11.810353679693337|
|           [0.0,0.0]|       26.8|11.810353679693337|
|           [0.0,0.0]|       80.3|11.810353679693337|
|           [0.0,1.0]|      -52.0| 11.86707476680698|
|           [0.0,1.0]|      -52.0| 11.86707476680698|
|           [0.0,1.0]|      -52.0| 11.86707476680698|
|           [0.0,1.0]|       -9.8| 11.86707476680698|
|           [0.0,1.0]|       -4.0| 11.86707476680698|
+--------------------+-----------+------------------+
only showing top 10 rows


In [16]:
pred_results.meanAbsoluteError, pred_results.meanSquaredError, pred_results.rootMeanSquaredError

(6.294027063194755, 104.3332892697012, 10.214366807086046)

### Handling Categorical Columns

In [18]:
from pyspark.ml.feature import StringIndexer

In [19]:
df_spark.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'pickup_longitude',
 'pickup_latitude',
 'RateCodeID',
 'store_and_fwd_flag',
 'dropoff_longitude',
 'dropoff_latitude',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount']

In [20]:
indexer = StringIndexer(inputCol='payment_type', outputCol='payment_type_indexed')

df_spark_indexed = indexer.fit(df_spark).transform(df_spark)
df_spark_indexed.show(10)

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude|RateCodeID|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|payment_type_indexed|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       2| 2015-01-15 19:05:39|  2015-01-15 19:23:42|              1|    

In [7]:
print(spark_job.version)
print(spark_job.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion())

4.2.0
3.5.0


In [8]:
import os
import pyspark

pyspark_path = os.path.dirname(pyspark.__file__)

print("PySpark location:")
print(pyspark_path)

print("\nChecking for Hadoop native files...")

for root, dirs, files in os.walk(pyspark_path):
    for file in files:
        if file.lower() in ["winutils.exe", "hadoop.dll", "hdfs.dll"]:
            print(os.path.join(root, file))

PySpark location:
c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark

Checking for Hadoop native files...


In [9]:
import os
import pyspark

jars_path = os.path.join(os.path.dirname(pyspark.__file__), "jars")

hadoop_jars = [
    f for f in os.listdir(jars_path)
    if f.startswith("hadoop-")
]

print("\nHadoop JARs:")
for jar in hadoop_jars:
    print(jar)


Hadoop JARs:
hadoop-client-api-3.5.0.jar
hadoop-client-runtime-3.5.0.jar


### File Formatting

In [6]:
df_spark.write.mode('overwrite').parquet('output/taxi_parquet')
df_from_parquet = spark_job.read.parquet('output/taxi_parquet')

# compare CSV vs Parquet size on disk
import os
print(f"CSV size: {os.path.getsize('2015.csv') / (1024*1024):.2f} MB")

# partition by a real column
df_spark.write.mode('overwrite').partitionBy('some_date_column').parquet('output/taxi_partitioned')

Py4JJavaError: An error occurred while calling o35.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs

In [ ]:
df_full = spark_job.read.option('header', 'true').option('inferSchema', 'true').csv('2015.csv')
df_full.count()   # time this, compare against Day 1's single-file count